In [2]:
spark

In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName("Spark_operation")\
.getOrCreate()

25/12/05 09:33:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
customer_data = [
"customer_id, name, city, state, country, registration_date, is_active"
"0,Customer_0,Bangalore, Karnataka, India, 2023-11-11, True",
"1,Customer_1,Hyderabad, Delhi, India, 2023-08-26,True",
"2,Customer_2,Ahmedabad,West Bengal, India,2023-06-23,True",
"3,Customer_3,Bangalore, Tamil Nadu, India, 2023-03-24, False",
"4,Customer_4, Bangalore, Gujarat, India, 2023-06-06, False",
"5,Customer_5,Delhi, Maharashtra, India, 2023-04-19,False",
]

In [5]:
data_rdd=spark.sparkContext.parallelize(customer_data)

In [6]:
data_rdd.collect()

['customer_id, name, city, state, country, registration_date, is_active0,Customer_0,Bangalore, Karnataka, India, 2023-11-11, True',
 '1,Customer_1,Hyderabad, Delhi, India, 2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal, India,2023-06-23,True',
 '3,Customer_3,Bangalore, Tamil Nadu, India, 2023-03-24, False',
 '4,Customer_4, Bangalore, Gujarat, India, 2023-06-06, False',
 '5,Customer_5,Delhi, Maharashtra, India, 2023-04-19,False']

In [7]:
data_rdd.getNumPartitions()

4

In [8]:
# RDD -Resillient distributed Datasets
# first()  -returns the first element of the rdd


In [9]:
header=data_rdd.first()

In [10]:
# filter() based on the condition  it is going to transform the dataset

In [11]:
data_rdd=data_rdd.filter(lambda x: x!=header)

In [12]:
data_rdd.collect()

['1,Customer_1,Hyderabad, Delhi, India, 2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal, India,2023-06-23,True',
 '3,Customer_3,Bangalore, Tamil Nadu, India, 2023-03-24, False',
 '4,Customer_4, Bangalore, Gujarat, India, 2023-06-06, False',
 '5,Customer_5,Delhi, Maharashtra, India, 2023-04-19,False']

In [13]:

def parse_row(row):
    
    fields = row.split(',')
    return (
     int(fields[0]),
     fields [1].strip(),
     fields [2].strip(),
     fields [3].strip(),
     fields [4].strip(),
     fields[5].strip(),
     fields [6] == 'True'

    )

parsed_rdd = data_rdd.map(parse_row)

In [14]:
parsed_rdd.collect()

[(1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

In [15]:
# advanced RDD operations

In [16]:
# extract a field with map -- city and state

In [17]:
city_state_rdd=parsed_rdd.map(lambda x:(x[1],x[2]))


In [18]:
city_state_rdd.collect()

[('Customer_1', 'Hyderabad'),
 ('Customer_2', 'Ahmedabad'),
 ('Customer_3', 'Bangalore'),
 ('Customer_4', 'Bangalore'),
 ('Customer_5', 'Delhi')]

In [19]:
# filter our active customers

active_cust_rdd=parsed_rdd.filter(lambda x:(x[6]==True))

In [20]:
active_cust_rdd.collect()

[(1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

In [21]:
# we can also use distinct() -- used in  transformations

In [22]:
cities_rdd=parsed_rdd.map(lambda x:x[2]).distinct()

In [23]:
cities_rdd.collect()

['Hyderabad', 'Ahmedabad', 'Delhi', 'Bangalore']

In [24]:
# take()

cities_rdd.take(3)

['Hyderabad', 'Ahmedabad', 'Delhi']

In [25]:
# reduce by key Transformation

In [26]:
# combine the values of eeach key by usig an associaive reduce function

In [28]:
customer_per_city=parsed_rdd.map(lambda x:(x[2],1)).reduceByKey(lambda a,b:a+b)

In [30]:
customer_per_city.collect()

[('Hyderabad', 1), ('Ahmedabad', 1), ('Delhi', 1), ('Bangalore', 2)]

In [32]:
# CountByValue()
cust_per_city=parsed_rdd.map(lambda x:x[2]).countByValue()

In [36]:
cust_per_city

defaultdict(int, {'Hyderabad': 1, 'Ahmedabad': 1, 'Bangalore': 2, 'Delhi': 1})

In [37]:
# Cities with active customer

active_cities = parsed_rdd. filter(lambda row:row[6]) \
.map(lambda row: row [2] ) \
.distinct()

In [38]:


active_cities.collect()

['Hyderabad', 'Ahmedabad']

In [42]:
active_cities.saveAsTextFile("/tmp/active_customers.csv")

In [44]:
! hadoop fs -ls /tmp/

Found 4 items
drwxr-xr-x   - root hadoop          0 2025-12-05 09:56 /tmp/active_customers.csv
drwxrwxrwt   - hdfs hadoop          0 2025-10-09 05:57 /tmp/hadoop-yarn
drwx-wx-wx   - hive hadoop          0 2025-10-09 05:57 /tmp/hive
-rw-r--r--   2 root hadoop         72 2025-12-03 19:11 /tmp/input_spark.txt
